In [ ]:
import pandas as pd
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric

DATASETS = {
    "adult": {
        "target": "Class-label",
        "versions": {
            "Original": "adult-clean.csv",
            "Synthetic": "adult_synthetic.csv",
            "Debiased": "adult_debiased.csv",
            "FLAI": "Adult-Flai-new-synthetic_fair_data.csv",
            "Decaf": "adult_decaf.csv",
        },
        "protected": {
            "gender": {
                "privileged": ["Male"],
                "unprivileged": ["Female"],
            },
            "race": {
                "privileged": ["White"],
                "unprivileged": "rest",
                "synthetic_code": 1,
            },
        },
        "favorable_label": 1,
        "unfavorable_label": 0,
    },

    "compas": {
        "target": "two_year_recid",
        "versions": {
            "Original": "compas-scores-two-years_clean.csv",
            "Synthetic": "compas_synthetic.csv",
            "Debiased": "compas_debiased.csv",
            "FLAI": "Compas-Flai-new-synthetic_fair_data.csv",
            "Decaf": "compas_decaf.csv",
        },
        "protected": {
            "sex": {
                "privileged": ["Male"],
                "unprivileged": ["Female"],
            },
            "race": {
                "privileged": ["Caucasian"],
                "unprivileged": "rest",
                "synthetic_code": 2,
            },
        },
        "favorable_label": 1,
        "unfavorable_label": 0,
    },

    "compas_violent": {
        "target": "two_year_recid",
        "versions": {
            "Original": "compas-scores-two-years-violent_clean.csv",
            "Synthetic": "compas_violent_synthetic.csv",
            "Debiased": "compas_violent_debiased.csv",
            "FLAI": "Compas-Violent-Flai-new-synthetic_fair_data.csv",
            "Decaf": "compas_violent_decaf.csv",
        },
        "protected": {
            "sex": {
                "privileged": ["Male"],
                "unprivileged": ["Female"],
            },
            "race": {
                "privileged": ["Caucasian"],
                "unprivileged": "rest",
                "synthetic_code": 2,
            },
        },
        "favorable_label": 1,
        "unfavorable_label": 0,
    },

    "diabetes": {
        "target": "readmitted",
        "versions": {
            "Original": "diabetes-clean.csv",
            "Synthetic": "diabetes_synthetic.csv",
            "Debiased": "diabetes_debiased.csv",
            # "FLAI": "diabetes_flai.csv",
            "Decaf": "diabetes_decaf.csv",
        },
        "protected": {
            "gender": {
                "privileged": ["Male"],
                "unprivileged": ["Female"],
            },
            "race": {
                "privileged": ["Caucasian"],
                "unprivileged": "rest",
                "synthetic_code": 2,
            },
        },
        "favorable_label": ">30",
        "unfavorable_label": "<30",
    },

    "dutch": {
        "target": "occupation",
        "versions": {
            "Original": "dutch.csv",
            "Synthetic": "dutch_synthetic.csv",
            "Debiased": "dutch_debiased.csv",
            "FLAI": "Dutch-Flai-new-synthetic_fair_data.csv",
            "Decaf": "dutch_decaf.csv",
        },
        "protected": {
            "sex": {
                "privileged": ["male"],
                "unprivileged": ["female"],
            },
        },
        "favorable_label": 1,
        "unfavorable_label": 0,
    },

    "law": {
        "target": "pass_bar",
        "versions": {
            "Original": "law_school_clean.csv",
            "Synthetic": "law_synthetic.csv",
            "Debiased": "law_debiased.csv",
            "FLAI": "Law-Flai-new-synthetic_fair_data.csv",
            "Decaf": "law_decaf.csv",
        },
        "protected": {
            "male": {
                "privileged": [1],
                "unprivileged": [0],
            },
            "race": {
                "privileged": ["White"],
                "unprivileged": "rest",
                "synthetic_code": 1,
            },
        },
        "favorable_label": 1,
        "unfavorable_label": 0,
    },
}


def compute_spd(df: pd.DataFrame,
                dataset_name: str,
                protected_attr: str,
                cfg: dict,
                version_name: str) -> float:
    target_col = cfg["target"]
    favorable_cfg = cfg["favorable_label"]

    # ---------- 1) LABEL → binary ----------
    if version_name == "Original":
        df["label_bin"] = (df[target_col] == favorable_cfg).astype(int)
    else:
        # all synthetic-like variants assumed numeric 0/1 with 1 = favorable
        df["label_bin"] = (df[target_col] == 1).astype(int)

    # ---------- 2) PROTECTED attribute groups (on original values) ----------
    prot_cfg = cfg["protected"][protected_attr]
    series = df[protected_attr]
    unique_vals = series.unique()
    synthetic_code = prot_cfg.get("synthetic_code")

    # Decide privileged/unprivileged values in terms of ORIGINAL values in df
    if version_name != "Original":
        # synthetic gender/sex: 0=Female, 1=Male
        if protected_attr in ["gender", "sex"]:
            priv_values = [1]
            unpriv_values = [0]
        elif synthetic_code is not None:
            priv_values = [synthetic_code]
            unpriv_values = [v for v in unique_vals if v not in priv_values]
        else:
            priv_raw = prot_cfg["privileged"]
            priv_values = priv_raw if isinstance(priv_raw, list) else [priv_raw]
            unpriv_cfg = prot_cfg.get("unprivileged")
            if unpriv_cfg is None or unpriv_cfg == "rest":
                unpriv_values = [v for v in unique_vals if v not in priv_values]
            else:
                unpriv_values = unpriv_cfg if isinstance(unpriv_cfg, list) else [unpriv_cfg]
    else:
        priv_raw = prot_cfg["privileged"]
        priv_values = priv_raw if isinstance(priv_raw, list) else [priv_raw]
        unpriv_cfg = prot_cfg.get("unprivileged")
        if unpriv_cfg is None or unpriv_cfg == "rest":
            unpriv_values = [v for v in unique_vals if v not in priv_values]
        else:
            unpriv_values = unpriv_cfg if isinstance(unpriv_cfg, list) else [unpriv_cfg]

    # ---------- 3) Encode protected_attr numerically for AIF360 ----------
    # AIF360 requires all df values numeric, so we map categories to ints,
    # but with consistent mapping for both series and group definitions.
    cats = list(unique_vals)
    mapping = {val: idx for idx, val in enumerate(cats)}

    df_local = pd.DataFrame({
        "prot": series.map(mapping),
        "label_bin": df["label_bin"].values,
    })

    priv_encoded = [mapping[v] for v in priv_values if v in mapping]
    unpriv_encoded = [mapping[v] for v in unpriv_values if v in mapping]

    privileged_groups = [{"prot": v} for v in priv_encoded]
    unprivileged_groups = [{"prot": v} for v in unpriv_encoded]

    # ---------- 4) Build BinaryLabelDataset and compute SPD ----------
    bld = BinaryLabelDataset(
        df=df_local,
        label_names=["label_bin"],
        protected_attribute_names=["prot"],
        favorable_label=1,
        unfavorable_label=0,
    )

    metric = BinaryLabelDatasetMetric(
        bld,
        privileged_groups=privileged_groups,
        unprivileged_groups=unprivileged_groups,
    )

    spd = metric.statistical_parity_difference()
    spd = -spd                      # privileged − unprivileged
    spd = round(spd, 4)
    return float(spd)


def main():
    rows = []

    for dataset_name, cfg in DATASETS.items():
        target_col = cfg["target"]

        print(f"\n=== Dataset: {dataset_name} (target: {target_col}) ===")

        for prot_attr in cfg["protected"].keys():
            print(f"\n  -> Protected attribute: {prot_attr}")
            row = {
                "Dataset": dataset_name,
                "Protected Attribute": prot_attr,
                "Target Attribute": target_col,
            }

            for version_name, filename in cfg["versions"].items():
                path = "data/" + filename
                df = pd.read_csv(path)
                spd = compute_spd(df, dataset_name, prot_attr, cfg, version_name)
                row[f"SPD-{version_name}"] = spd

                print(f"     {version_name:<9} | SPD = {spd:.6f}")

            rows.append(row)

    df_out = pd.DataFrame(rows, columns=[
        "Dataset",
        "Protected Attribute",
        "Target Attribute",
        "SPD-Original",
        "SPD-Synthetic",
        "SPD-Debiased",
        "SPD-FLAI",
        "SPD-Decaf",
    ])

    print("\n\n=== Final SPD Table ===")
    print(df_out.to_string(index=False))

    # df_out.to_excel("SPD_AllDatasets.xlsx", index=False)
    # print("\nDone. Results written to SPD_AllDatasets.xlsx")


if __name__ == "__main__":
    main()